In [1]:
import pandas as pd
from pathlib import Path

# Locate the project data regardless of whether VS Code runs from the
# project folder or its parent folder.
candidates = [
    Path("data/uac_data.csv"),
    Path("UAC_Forecasting_Project/data/uac_data.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        f"Could not find uac_data.csv. Current directory: {Path.cwd()}"
    )

df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Dataset shape: {df.shape}")
display(df.head())


Loaded: UAC_Forecasting_Project\data\uac_data.csv
Dataset shape: (1170, 6)


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0


In [2]:
# Prepare the columns needed for feature engineering
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_columns = [
    "Children in HHS Care",
    "Children transferred out of CBP custody",
    "Children discharged from HHS Care",
]

for col in numeric_columns:
    if col not in df.columns:
        raise KeyError(f"Required column not found: {col}")
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

print("Prepared data:")
display(df[["Date"] + numeric_columns].head())


Prepared data:


,Date,Children in HHS Care,Children transferred out of CBP custody,Children discharged from HHS Care
0,2023-01-12,6566.0,34.0,436.0
1,2023-01-22,7122.0,39.0,227.0
2,2023-01-23,7280.0,39.0,181.0
3,2023-01-24,7433.0,47.0,175.0
4,2023-01-25,7538.0,41.0,180.0


In [3]:
# Lag features
target = "Children in HHS Care"

df["lag1"] = df[target].shift(1)
df["lag7"] = df[target].shift(7)
df["lag14"] = df[target].shift(14)

display(df[["Date", target, "lag1", "lag7", "lag14"]].head(20))


,Date,Children in HHS Care,lag1,lag7,lag14
0,2023-01-12,6566.0,NaN,NaN,NaN
1,2023-01-22,7122.0,6566.0,NaN,NaN
2,2023-01-23,7280.0,7122.0,NaN,NaN
3,2023-01-24,7433.0,7280.0,NaN,NaN
4,2023-01-25,7538.0,7433.0,NaN,NaN
5,2023-01-29,7472.0,7538.0,NaN,NaN
6,2023-01-30,7743.0,7472.0,NaN,NaN
7,2023-01-31,7803.0,7743.0,6566.0,NaN
8,2023-02-01,7903.0,7803.0,7122.0,NaN
9,2023-02-02,7879.0,7903.0,7280.0,NaN


In [4]:
# Rolling mean features
df["roll7"] = df[target].rolling(window=7, min_periods=1).mean()
df["roll14"] = df[target].rolling(window=14, min_periods=1).mean()

display(df[["Date", target, "roll7", "roll14"]].head(20))


,Date,Children in HHS Care,roll7,roll14
0,2023-01-12,6566.0,6566.000000,6566.000000
1,2023-01-22,7122.0,6844.000000,6844.000000
2,2023-01-23,7280.0,6989.333333,6989.333333
3,2023-01-24,7433.0,7100.250000,7100.250000
4,2023-01-25,7538.0,7187.800000,7187.800000
5,2023-01-29,7472.0,7235.166667,7235.166667
6,2023-01-30,7743.0,7307.714286,7307.714286
7,2023-01-31,7803.0,7484.428571,7369.625000
8,2023-02-01,7903.0,7596.000000,7428.888889
9,2023-02-02,7879.0,7681.571429,7473.900000


In [5]:
# Net pressure feature
df["net_pressure"] = (
    df["Children transferred out of CBP custody"]
    - df["Children discharged from HHS Care"]
)

display(
    df[
        [
            "Date",
            "Children transferred out of CBP custody",
            "Children discharged from HHS Care",
            "net_pressure",
        ]
    ].head(20)
)


,Date,Children transferred out of CBP custody,Children discharged from HHS Care,net_pressure
0,2023-01-12,34.0,436.0,-402.0
1,2023-01-22,39.0,227.0,-188.0
2,2023-01-23,39.0,181.0,-142.0
3,2023-01-24,47.0,175.0,-128.0
4,2023-01-25,41.0,180.0,-139.0
5,2023-01-29,11.0,303.0,-292.0
6,2023-01-30,29.0,196.0,-167.0
7,2023-01-31,36.0,158.0,-122.0
8,2023-02-01,27.0,231.0,-204.0
9,2023-02-02,23.0,298.0,-275.0


In [6]:
# Calendar features
df["dayofweek"] = df["Date"].dt.dayofweek
df["month"] = df["Date"].dt.month

display(df[["Date", "dayofweek", "month"]].head(20))


,Date,dayofweek,month
0,2023-01-12,3.0,1.0
1,2023-01-22,6.0,1.0
2,2023-01-23,0.0,1.0
3,2023-01-24,1.0,1.0
4,2023-01-25,2.0,1.0
5,2023-01-29,6.0,1.0
6,2023-01-30,0.0,1.0
7,2023-01-31,1.0,1.0
8,2023-02-01,2.0,2.0
9,2023-02-02,3.0,2.0


In [7]:
# Remove rows that cannot be used for lag-based modeling.
# The first 14 rows do not have all lag14 values.
feature_columns = [
    "lag1", "lag7", "lag14", "roll7", "roll14",
    "net_pressure", "dayofweek", "month"
]

df_model = df.dropna(subset=[target, "Date"] + feature_columns).copy()

print(f"Original rows: {len(df)}")
print(f"Rows after feature preparation: {len(df_model)}")
print(f"Missing values in model features:")
display(df_model[feature_columns + [target]].isna().sum())


Original rows: 1170
Rows after feature preparation: 706
Missing values in model features:


lag1                    0
lag7                    0
lag14                   0
roll7                   0
roll14                  0
net_pressure            0
dayofweek               0
month                   0
Children in HHS Care    0
dtype: int64

In [8]:
# Save processed data inside the actual project data folder.
project_data_dir = Path("data")
if not project_data_dir.exists():
    project_data_dir = Path("UAC_Forecasting_Project/data")

project_data_dir.mkdir(parents=True, exist_ok=True)
processed_path = project_data_dir / "processed_data.csv"

# Save the full engineered dataframe so later notebooks can access all rows.
df.to_csv(processed_path, index=False)

print(f"Processed data saved to: {processed_path.resolve()}")
print(f"Final shape: {df.shape}")


Processed data saved to: C:\Users\kapta\Downloads\UAC-Forecasting-Project-main\UAC-Forecasting-Project-main\UAC_Forecasting_Project\data\processed_data.csv
Final shape: (1170, 14)
